In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from segment_anything import sam_model_registry
import os
from glob import glob
from scipy import ndimage

# Your class info with complete color map
CLASS_INFO = {
    0: {"name": "background", "rgb": [0, 0, 0]},
    1: {"name": "airplane", "rgb": [166, 202, 240]},
    2: {"name": "bare soil", "rgb": [128, 128, 0]},
    3: {"name": "buildings", "rgb": [0, 0, 128]},
    4: {"name": "cars", "rgb": [255, 0, 0]},
    5: {"name": "chaparral", "rgb": [0, 128, 0]},
    6: {"name": "court", "rgb": [128, 0, 0]},
    7: {"name": "dock", "rgb": [255, 233, 233]},
    8: {"name": "field", "rgb": [160, 160, 164]},
    9: {"name": "grass", "rgb": [0, 128, 128]},
    10: {"name": "mobile home", "rgb": [90, 87, 255]},
    11: {"name": "pavement", "rgb": [255, 255, 0]},
    12: {"name": "sand", "rgb": [255, 192, 0]},
    13: {"name": "sea", "rgb": [0, 0, 255]},
    14: {"name": "ship", "rgb": [255, 0, 192]},
    15: {"name": "tanks", "rgb": [128, 0, 128]},
    16: {"name": "trees", "rgb": [0, 255, 0]},
    17: {"name": "water", "rgb": [0, 255, 255]}
}
NUM_CLASSES = len(CLASS_INFO)

class MultiClassSAMWrapper(nn.Module):
    def __init__(self, sam_model, num_classes):
        super().__init__()
        self.sam = sam_model
        self.num_classes = num_classes
        self.multi_class_head = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(256, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, 1)
        )
        for param in self.sam.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        with torch.no_grad():
            image_embeddings = self.sam.image_encoder(x)
        logits = self.multi_class_head(image_embeddings)
        return logits

class ConfidenceNet(nn.Module):
    def __init__(self, num_classes=18, feature_dim=256):
        super().__init__()
        self.num_classes = num_classes
        
        # Input branches
        self.probability_encoder = nn.Sequential(
            nn.Conv2d(num_classes, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        
        self.point_encoder = nn.Sequential(
            nn.Conv2d(num_classes, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        
        self.geometric_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        
        # Main fusion network
        self.fusion_net = nn.Sequential(
            nn.Conv2d(32 + 32 + 32, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid()
        )
        
    def forward(self, probability_maps, point_maps, geometric_features):
        prob_features = self.probability_encoder(probability_maps)
        point_features = self.point_encoder(point_maps)
        geom_features = self.geometric_encoder(geometric_features)
        
        fused = torch.cat([prob_features, point_features, geom_features], dim=1)
        confidence_map = self.fusion_net(fused)
        return confidence_map

def preprocess_image(image):
    """Preprocess image for SAM"""
    input_image = cv2.resize(image, (1024, 1024))
    input_image = input_image.astype(np.float32) / 255.0
    input_image = (input_image - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
    return torch.from_numpy(input_image).permute(2, 0, 1).float()

def save_sam_predictions(image_dir, mask_dir, output_dir, sam_model, device):
    """Pre-compute and save all SAM predictions"""
    os.makedirs(output_dir, exist_ok=True)
    image_files = sorted(glob(os.path.join(image_dir, "*.png")))
    
    print(f"Pre-computing SAM predictions for {len(image_files)} images...")
    
    for i, image_path in enumerate(image_files):
        if i % 50 == 0:
            print(f"Processing {i}/{len(image_files)}...")
        
        base_name = os.path.basename(image_path).split('.')[0]
        
        # Load image
        image = cv2.imread(image_path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        original_size = image.shape[:2]
        
        # Get SAM prediction
        with torch.no_grad():
            input_tensor = preprocess_image(image).unsqueeze(0).to(device)
            logits = sam_model(input_tensor)
            probabilities = F.softmax(logits, dim=1)
            
            # Resize to original size
            logits_resized = F.interpolate(logits, size=original_size, mode='bilinear', align_corners=False)
            probabilities_resized = F.interpolate(probabilities, size=original_size, mode='bilinear', align_corners=False)
            
            prediction = torch.argmax(logits_resized, dim=1).squeeze(0).cpu().numpy()
            probabilities_np = probabilities_resized.squeeze(0).cpu().numpy()
        
        # Save predictions
        pred_save_path = os.path.join(output_dir, f"{base_name}_pred.npy")
        prob_save_path = os.path.join(output_dir, f"{base_name}_prob.npy")
        
        np.save(pred_save_path, prediction)
        np.save(prob_save_path, probabilities_np)
    
    print("SAM predictions pre-computation completed!")

class ConfidenceDataset(Dataset):
    def __init__(self, image_dir, mask_dir, point_dir, prediction_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.point_dir = point_dir
        self.prediction_dir = prediction_dir
        
        self.image_files = sorted(glob(os.path.join(image_dir, "*.png")))
        
    def __len__(self):
        return len(self.image_files)
    
    def calculate_geometric_features(self, prediction_mask):
        """Calculate boundary smoothness and region coherence"""
        features = np.zeros((3, prediction_mask.shape[0], prediction_mask.shape[1]))
        
        # 1. Boundary map
        boundary = np.zeros_like(prediction_mask, dtype=bool)
        h, w = prediction_mask.shape
        for i in range(1, h-1):
            for j in range(1, w-1):
                current = prediction_mask[i, j]
                if (prediction_mask[i-1, j] != current or prediction_mask[i+1, j] != current or
                    prediction_mask[i, j-1] != current or prediction_mask[i, j+1] != current):
                    boundary[i, j] = True
        
        features[0] = boundary.astype(np.float32)
        
        # 2. Distance to boundary
        distance = ndimage.distance_transform_edt(~boundary)
        features[1] = distance / (np.max(distance) + 1e-8)
        
        # 3. Region size feature
        labeled, num_regions = ndimage.label(prediction_mask)
        region_sizes = np.array([np.sum(labeled == i) for i in range(1, num_regions + 1)])
        if len(region_sizes) > 0:
            size_map = np.zeros_like(prediction_mask, dtype=np.float32)
            for i in range(1, num_regions + 1):
                size_map[labeled == i] = region_sizes[i-1] / prediction_mask.size
            features[2] = size_map
        
        return features
    
    def create_point_map(self, base_name, prediction_shape):
        """Create point guidance map from point annotations - IGNORING (100,100,100) pixels"""
        point_map = np.zeros((NUM_CLASSES, prediction_shape[0], prediction_shape[1]))
        
        point_path = os.path.join(self.point_dir, f"{base_name}.png")
        if os.path.exists(point_path):
            point_mask = cv2.imread(point_path)
            if point_mask is not None:
                point_mask = cv2.cvtColor(point_mask, cv2.COLOR_BGR2RGB)
                
                # IGNORE pixels with color (100, 100, 100)
                ignore_color = np.array([100, 100, 100])
                non_ignore_mask = ~np.all(point_mask == ignore_color, axis=-1)
                
                for class_id, info in CLASS_INFO.items():
                    if class_id == 0:  # Skip background
                        continue
                    target_color = np.array(info["rgb"])
                    class_points = np.all(point_mask == target_color, axis=-1) & non_ignore_mask
                    
                    if np.any(class_points):
                        # Create influence map with exponential decay from points
                        distance = ndimage.distance_transform_edt(~class_points)
                        influence = np.exp(-distance / 10.0)  # Adjust decay rate as needed
                        point_map[class_id] = influence
        
        return point_map
    
    def __getitem__(self, idx):
        image_path = self.image_files[idx]
        base_name = os.path.basename(image_path).split('.')[0]
        
        try:
            # Load PRE-COMPUTED SAM predictions
            pred_path = os.path.join(self.prediction_dir, f"{base_name}_pred.npy")
            prob_path = os.path.join(self.prediction_dir, f"{base_name}_prob.npy")
            
            prediction = np.load(pred_path)
            probability_maps = np.load(prob_path)
            
            # Load ground truth
            mask_path = os.path.join(self.mask_dir, f"{base_name}.png")
            ground_truth = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            if ground_truth is None:
                raise ValueError(f"Could not load ground truth mask: {mask_path}")
            
            # Ensure same shape
            if ground_truth.shape != prediction.shape:
                ground_truth = cv2.resize(ground_truth, (prediction.shape[1], prediction.shape[0]), 
                                        interpolation=cv2.INTER_NEAREST)
            
            # Create confidence labels (1 where correct, 0 where wrong)
            confidence_labels = (prediction == ground_truth).astype(np.float32)
            
            # Create input features
            geometric_features = self.calculate_geometric_features(prediction)
            point_maps = self.create_point_map(base_name, prediction.shape)
            
            # Convert to tensors
            probability_tensor = torch.from_numpy(probability_maps).float()
            point_tensor = torch.from_numpy(point_maps).float()
            geometric_tensor = torch.from_numpy(geometric_features).float()
            confidence_tensor = torch.from_numpy(confidence_labels).float().unsqueeze(0)
            
            return probability_tensor, point_tensor, geometric_tensor, confidence_tensor
            
        except Exception as e:
            print(f"Error loading {base_name}: {e}")
            # Return zeros if there's an error (with appropriate shape)
            dummy_shape = (512, 512)  # Adjust based on your typical image size
            return (
                torch.zeros((NUM_CLASSES, *dummy_shape)),
                torch.zeros((NUM_CLASSES, *dummy_shape)),
                torch.zeros((3, *dummy_shape)),
                torch.zeros((1, *dummy_shape))
            )

def train_confidence_net():
    # Configuration
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Paths - UPDATE THESE PATHS to your actual directories
    train_image_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"  # Your 640 training images
    train_mask_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"    # Your dense masks
    train_point_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"   # Your point annotations
    prediction_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_sam_predictions" # Where to save predictions
    
    # Load your fine-tuned SAM
    print("Loading SAM model...")
    sam_checkpoint = "/home/iiitdmk-param/Desktop/sam_vit_b.pth"
    model_type = "vit_b"
    pretrained_model_path = 'best_multi_class_sam.pth'
    
    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
    model = MultiClassSAMWrapper(sam, NUM_CLASSES)
    
    # Load your fine-tuned weights
    checkpoint = torch.load(pretrained_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    # Step 1: Pre-compute SAM predictions
    print("Step 1: Pre-computing SAM predictions...")
    save_sam_predictions(train_image_dir, train_mask_dir, prediction_dir, model, device)
    
    # Step 2: Create dataset and dataloader
    print("Step 2: Creating dataset...")
    dataset = ConfidenceDataset(train_image_dir, train_mask_dir, train_point_dir, prediction_dir)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4)
    
    # Step 3: Initialize and train ConfidenceNet
    print("Step 3: Training ConfidenceNet...")
    confidence_net = ConfidenceNet(num_classes=NUM_CLASSES)
    confidence_net.to(device)
    
    optimizer = optim.Adam(confidence_net.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    num_epochs = 50
    for epoch in range(num_epochs):
        confidence_net.train()
        running_loss = 0.0
        
        for batch_idx, (prob_maps, point_maps, geom_features, confidence_labels) in enumerate(dataloader):
            prob_maps = prob_maps.to(device)
            point_maps = point_maps.to(device)
            geom_features = geom_features.to(device)
            confidence_labels = confidence_labels.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            predicted_confidence = confidence_net(prob_maps, point_maps, geom_features)
            
            # Calculate loss
            loss = criterion(predicted_confidence, confidence_labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}')
        
        avg_loss = running_loss / len(dataloader)
        print(f'Epoch {epoch} completed. Average Loss: {avg_loss:.4f}')
        
        # Save checkpoint
        if epoch % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': confidence_net.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, f'confidence_net_epoch_{epoch}.pth')
    
    # Save final model
    torch.save(confidence_net.state_dict(), 'confidence_net_final.pth')
    print("ConfidenceNet training completed!")

if __name__ == "__main__":
    train_confidence_net()

Using device: cuda
Loading SAM model...


/home/iiitdmk-param/AMB/Proposed_3/SAM/segment-anything-main/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)
/tmp

Step 1: Pre-computing SAM predictions...
Pre-computing SAM predictions for 630 images...
Processing 0/630...
Processing 50/630...
Processing 100/630...
Processing 150/630...
Processing 200/630...
Processing 250/630...
Processing 300/630...
Processing 350/630...
Processing 400/630...
Processing 450/630...
Processing 500/630...
Processing 550/630...
Processing 600/630...
SAM predictions pre-computation completed!
Step 2: Creating dataset...
Step 3: Training ConfidenceNet...
Epoch 0, Batch 0, Loss: 0.2601
Epoch 0, Batch 10, Loss: 0.1581
Epoch 0, Batch 20, Loss: 0.1298
Epoch 0, Batch 30, Loss: 0.1043
Epoch 0, Batch 40, Loss: 0.0878
Epoch 0, Batch 50, Loss: 0.0718
Epoch 0, Batch 60, Loss: 0.0627
Epoch 0, Batch 70, Loss: 0.0489
Epoch 0, Batch 80, Loss: 0.0442
Epoch 0, Batch 90, Loss: 0.0348
Epoch 0, Batch 100, Loss: 0.0290
Epoch 0, Batch 110, Loss: 0.0250
Epoch 0, Batch 120, Loss: 0.0225
Epoch 0, Batch 130, Loss: 0.0195
Epoch 0, Batch 140, Loss: 0.0187
Epoch 0, Batch 150, Loss: 0.0163
Epoch 

In [5]:
import torch
import numpy as np
import cv2
import os
from glob import glob
import pandas as pd
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F

def calculate_iou(pred, target, num_classes):
    """Calculate IoU for each class"""
    ious = []
    for class_id in range(num_classes):
        pred_mask = (pred == class_id)
        target_mask = (target == class_id)
        
        intersection = np.logical_and(pred_mask, target_mask).sum()
        union = np.logical_or(pred_mask, target_mask).sum()
        
        if union == 0:
            ious.append(1.0)  # If both are empty, IoU is 1
        else:
            ious.append(intersection / union)
    
    return ious

def calculate_miou(pred, target, num_classes):
    """Calculate mean IoU"""
    ious = calculate_iou(pred, target, num_classes)
    return np.mean(ious)

def test_confidence_net_on_train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load trained ConfidenceNet
    confidence_net = ConfidenceNet(num_classes=NUM_CLASSES)
    confidence_net.load_state_dict(torch.load('confidence_net_final.pth', map_location=device))
    confidence_net.to(device)
    confidence_net.eval()
    
    # Paths for your training data
    train_image_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
    train_mask_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"
    train_point_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
    prediction_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_sam_predictions"
    
    # Get all training images
    image_files = sorted(glob(os.path.join(train_image_dir, "*.png")))
    print(f"Testing on {len(image_files)} training images...")
    
    results = []
    
    for i, image_path in enumerate(image_files):
        if i % 50 == 0:
            print(f"Processed {i}/{len(image_files)} images...")
        
        base_name = os.path.basename(image_path).split('.')[0]
        
        try:
            # Load pre-computed SAM predictions
            pred_path = os.path.join(prediction_dir, f"{base_name}_pred.npy")
            prob_path = os.path.join(prediction_dir, f"{base_name}_prob.npy")
            
            if not os.path.exists(pred_path) or not os.path.exists(prob_path):
                print(f"⚠️ Missing prediction files for {base_name}")
                continue
            
            prediction = np.load(pred_path)
            probability_maps = np.load(prob_path)
            
            # Load ground truth
            mask_path = os.path.join(train_mask_dir, f"{base_name}.png")
            ground_truth = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            if ground_truth is None:
                print(f"⚠️ Could not load ground truth for {base_name}")
                continue
            
            # Ensure same shape
            if ground_truth.shape != prediction.shape:
                ground_truth = cv2.resize(ground_truth, (prediction.shape[1], prediction.shape[0]), 
                                        interpolation=cv2.INTER_NEAREST)
            
            # Calculate actual mIoU
            miou = calculate_miou(prediction, ground_truth, NUM_CLASSES)
            
            # Calculate geometric features
            dataset = ConfidenceDataset(train_image_dir, train_mask_dir, train_point_dir, prediction_dir)
            geometric_features = dataset.calculate_geometric_features(prediction)
            point_maps = dataset.create_point_map(base_name, prediction.shape)
            
            # Convert to tensors
            probability_tensor = torch.from_numpy(probability_maps).float().unsqueeze(0).to(device)
            point_tensor = torch.from_numpy(point_maps).float().unsqueeze(0).to(device)
            geometric_tensor = torch.from_numpy(geometric_features).float().unsqueeze(0).to(device)
            
            # Get confidence scores from ConfidenceNet
            with torch.no_grad():
                confidence_scores = confidence_net(probability_tensor, point_tensor, geometric_tensor)
                confidence_map = confidence_scores.squeeze().cpu().numpy()
            
            # Calculate average confidence
            avg_confidence = np.mean(confidence_map)
            max_confidence = np.max(confidence_map)
            min_confidence = np.min(confidence_map)
            
            # Calculate percentage of confident pixels (threshold = 0.7)
            confident_pixels = np.sum(confidence_map > 0.7)
            total_pixels = confidence_map.size
            confident_ratio = confident_pixels / total_pixels
            
            # Store results
            result = {
                'image_name': base_name,
                'confidence_net_output': avg_confidence,
                'actual_miou': miou,
                'max_confidence': max_confidence,
                'min_confidence': min_confidence,
                'confident_pixels_ratio': confident_ratio,
                'confident_pixels_count': confident_pixels,
                'total_pixels': total_pixels
            }
            
            # Add per-class IoU
            class_ious = calculate_iou(prediction, ground_truth, NUM_CLASSES)
            for class_id, iou in enumerate(class_ious):
                if iou > 0:  # Only include classes that appear
                    result[f'class_{class_id}_iou'] = iou
                    result[f'class_{class_id}_name'] = CLASS_INFO[class_id]['name']
            
            results.append(result)
            
            print(f"📊 {base_name}: Confidence={avg_confidence:.3f}, mIoU={miou:.3f}")
            
        except Exception as e:
            print(f"⚠️ Error processing {base_name}: {e}")
            continue
    
    # Create DataFrame and save to CSV
    df = pd.DataFrame(results)
    
    # Calculate correlation between confidence and mIoU
    correlation = df['confidence_net_output'].corr(df['actual_miou'])
    print(f"\n📈 Correlation between ConfidenceNet output and mIoU: {correlation:.3f}")
    
    # Save detailed results
    output_csv = "confidence_vs_miou_results.csv"
    df.to_csv(output_csv, index=False)
    print(f"✅ Results saved to {output_csv}")
    
    # Print summary statistics
    print(f"\n📊 Summary Statistics:")
    print(f"Average ConfidenceNet output: {df['confidence_net_output'].mean():.3f}")
    print(f"Average mIoU: {df['actual_miou'].mean():.3f}")
    print(f"Confidence range: {df['confidence_net_output'].min():.3f} - {df['confidence_net_output'].max():.3f}")
    print(f"mIoU range: {df['actual_miou'].min():.3f} - {df['actual_miou'].max():.3f}")
    
    # Analyze high vs low confidence examples
    high_conf_threshold = df['confidence_net_output'].quantile(0.75)
    low_conf_threshold = df['confidence_net_output'].quantile(0.25)
    
    high_conf_miou = df[df['confidence_net_output'] >= high_conf_threshold]['actual_miou'].mean()
    low_conf_miou = df[df['confidence_net_output'] <= low_conf_threshold]['actual_miou'].mean()
    
    print(f"\n🎯 Confidence vs Performance:")
    print(f"High confidence (top 25%): mIoU = {high_conf_miou:.3f}")
    print(f"Low confidence (bottom 25%): mIoU = {low_conf_miou:.3f}")
    print(f"Performance gap: {high_conf_miou - low_conf_miou:.3f}")
    
    return df

def analyze_class_wise_performance(df):
    """Analyze performance for each class"""
    print(f"\n🎯 Class-wise Analysis:")
    
    # Extract class-wise IoU columns
    class_columns = [col for col in df.columns if col.startswith('class_') and col.endswith('_iou')]
    
    class_performance = []
    for col in class_columns:
        class_id = int(col.split('_')[1])
        class_name = CLASS_INFO[class_id]['name']
        
        # Get non-null IoU values for this class
        class_ious = df[col].dropna()
        if len(class_ious) > 0:
            avg_iou = class_ious.mean()
            class_performance.append({
                'class_id': class_id,
                'class_name': class_name,
                'avg_iou': avg_iou,
                'samples_count': len(class_ious)
            })
    
    # Sort by average IoU
    class_performance.sort(key=lambda x: x['avg_iou'], reverse=True)
    
    print(f"\n🏆 Best performing classes:")
    for perf in class_performance[:5]:
        print(f"  {perf['class_name']}: IoU={perf['avg_iou']:.3f} ({perf['samples_count']} samples)")
    
    print(f"\n📉 Worst performing classes:")
    for perf in class_performance[-5:]:
        print(f"  {perf['class_name']}: IoU={perf['avg_iou']:.3f} ({perf['samples_count']} samples)")
    
    # Save class-wise analysis
    class_df = pd.DataFrame(class_performance)
    class_df.to_csv("class_wise_performance.csv", index=False)
    print(f"✅ Class-wise analysis saved to class_wise_performance.csv")

if __name__ == "__main__":
    # Test ConfidenceNet on training data
    results_df = test_confidence_net_on_train()
    
    # Analyze class-wise performance
    analyze_class_wise_performance(results_df)
    
    print(f"\n🎉 Analysis completed!")
    print(f"Check 'confidence_vs_miou_results.csv' for detailed results")
    print(f"Check 'class_wise_performance.csv' for class-wise analysis")

/tmp/ipykernel_38155/4016134043.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  confidence_net.load_state_dict(torch.load('confidence_net_final.pth', map_location=devic

Testing on 630 training images...
Processed 0/630 images...
📊 1: Confidence=0.001, mIoU=0.611
📊 10: Confidence=0.001, mIoU=0.889
📊 100: Confidence=0.001, mIoU=0.667
📊 101: Confidence=0.001, mIoU=0.667
📊 102: Confidence=0.001, mIoU=0.611
📊 103: Confidence=0.001, mIoU=0.667
📊 104: Confidence=0.001, mIoU=0.556
📊 105: Confidence=0.001, mIoU=0.667
📊 106: Confidence=0.001, mIoU=0.611
📊 107: Confidence=0.001, mIoU=0.611
📊 108: Confidence=0.001, mIoU=0.722
📊 109: Confidence=0.001, mIoU=0.722
📊 11: Confidence=0.001, mIoU=0.833
📊 110: Confidence=0.001, mIoU=0.556
📊 111: Confidence=0.001, mIoU=0.667
📊 112: Confidence=0.001, mIoU=0.611
📊 113: Confidence=0.001, mIoU=0.611
📊 114: Confidence=0.001, mIoU=0.444
📊 115: Confidence=0.001, mIoU=0.500
📊 116: Confidence=0.001, mIoU=0.667
📊 117: Confidence=0.001, mIoU=0.667
📊 118: Confidence=0.001, mIoU=0.556
📊 119: Confidence=0.001, mIoU=0.667
📊 12: Confidence=0.001, mIoU=0.778
📊 120: Confidence=0.001, mIoU=0.667
📊 121: Confidence=0.001, mIoU=0.611
📊 122: Co

In [3]:
import torch
import numpy as np
import cv2
import os
from glob import glob
import pandas as pd
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F

def calculate_iou(pred, target, num_classes):
    """Calculate IoU for each class"""
    ious = []
    for class_id in range(num_classes):
        pred_mask = (pred == class_id)
        target_mask = (target == class_id)
        
        intersection = np.logical_and(pred_mask, target_mask).sum()
        union = np.logical_or(pred_mask, target_mask).sum()
        
        if union == 0:
            ious.append(1.0)  # If both are empty, IoU is 1
        else:
            ious.append(intersection / union)
    
    return ious

def calculate_miou(pred, target, num_classes):
    """Calculate mean IoU"""
    ious = calculate_iou(pred, target, num_classes)
    return np.mean(ious)

def test_confidence_net_on_train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load trained ConfidenceNet
    confidence_net = ConfidenceNet(num_classes=NUM_CLASSES)
    confidence_net.load_state_dict(torch.load('confidence_net_final.pth', map_location=device))
    confidence_net.to(device)
    confidence_net.eval()
    
    # Paths for your training data
    train_image_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
    train_mask_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"
    train_point_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
    prediction_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_sam_predictions"
    
    # Get all training images
    image_files = sorted(glob(os.path.join(train_image_dir, "*.png")))
    print(f"Testing on {len(image_files)} training images...")
    
    results = []
    
    for i, image_path in enumerate(image_files):
        if i % 50 == 0:
            print(f"Processed {i}/{len(image_files)} images...")
        
        base_name = os.path.basename(image_path).split('.')[0]
        
        try:
            # Load pre-computed SAM predictions
            pred_path = os.path.join(prediction_dir, f"{base_name}_pred.npy")
            prob_path = os.path.join(prediction_dir, f"{base_name}_prob.npy")
            
            if not os.path.exists(pred_path) or not os.path.exists(prob_path):
                print(f"⚠️ Missing prediction files for {base_name}")
                continue
            
            prediction = np.load(pred_path)
            probability_maps = np.load(prob_path)
            
            # Load ground truth
            mask_path = os.path.join(train_mask_dir, f"{base_name}.png")
            ground_truth = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            if ground_truth is None:
                print(f"⚠️ Could not load ground truth for {base_name}")
                continue
            
            # Ensure same shape
            if ground_truth.shape != prediction.shape:
                ground_truth = cv2.resize(ground_truth, (prediction.shape[1], prediction.shape[0]), 
                                        interpolation=cv2.INTER_NEAREST)
            
            # Calculate actual mIoU
            miou = calculate_miou(prediction, ground_truth, NUM_CLASSES)
            
            # Calculate geometric features
            dataset = ConfidenceDataset(train_image_dir, train_mask_dir, train_point_dir, prediction_dir)
            geometric_features = dataset.calculate_geometric_features(prediction)
            point_maps = dataset.create_point_map(base_name, prediction.shape)
            
            # Convert to tensors
            probability_tensor = torch.from_numpy(probability_maps).float().unsqueeze(0).to(device)
            point_tensor = torch.from_numpy(point_maps).float().unsqueeze(0).to(device)
            geometric_tensor = torch.from_numpy(geometric_features).float().unsqueeze(0).to(device)
            
            # Get confidence scores from ConfidenceNet
            with torch.no_grad():
                confidence_scores = confidence_net(probability_tensor, point_tensor, geometric_tensor)
                confidence_map = confidence_scores.squeeze().cpu().numpy()
            
            # Calculate average confidence
            avg_confidence = np.mean(confidence_map)
            max_confidence = np.max(confidence_map)
            min_confidence = np.min(confidence_map)
            
            # Calculate percentage of confident pixels (threshold = 0.7)
            confident_pixels = np.sum(confidence_map > 0.7)
            total_pixels = confidence_map.size
            confident_ratio = confident_pixels / total_pixels
            
            # Store results
            result = {
                'image_name': base_name,
                'confidence_net_output': avg_confidence,
                'actual_miou': miou,
                'max_confidence': max_confidence,
                'min_confidence': min_confidence,
                'confident_pixels_ratio': confident_ratio,
                'confident_pixels_count': confident_pixels,
                'total_pixels': total_pixels
            }
            
            # Add per-class IoU
            class_ious = calculate_iou(prediction, ground_truth, NUM_CLASSES)
            for class_id, iou in enumerate(class_ious):
                if iou > 0:  # Only include classes that appear
                    result[f'class_{class_id}_iou'] = iou
                    result[f'class_{class_id}_name'] = CLASS_INFO[class_id]['name']
            
            results.append(result)
            
            print(f"📊 {base_name}: Confidence={avg_confidence:.3f}, mIoU={miou:.3f}")
            
        except Exception as e:
            print(f"⚠️ Error processing {base_name}: {e}")
            continue
    
    # Create DataFrame and save to CSV
    df = pd.DataFrame(results)
    
    # Calculate correlation between confidence and mIoU
    correlation = df['confidence_net_output'].corr(df['actual_miou'])
    print(f"\n📈 Correlation between ConfidenceNet output and mIoU: {correlation:.3f}")
    
    # Save detailed results
    output_csv = "confidence_vs_miou_results.csv"
    df.to_csv(output_csv, index=False)
    print(f"✅ Results saved to {output_csv}")
    
    # Print summary statistics
    print(f"\n📊 Summary Statistics:")
    print(f"Average ConfidenceNet output: {df['confidence_net_output'].mean():.3f}")
    print(f"Average mIoU: {df['actual_miou'].mean():.3f}")
    print(f"Confidence range: {df['confidence_net_output'].min():.3f} - {df['confidence_net_output'].max():.3f}")
    print(f"mIoU range: {df['actual_miou'].min():.3f} - {df['actual_miou'].max():.3f}")
    
    # Analyze high vs low confidence examples
    high_conf_threshold = df['confidence_net_output'].quantile(0.75)
    low_conf_threshold = df['confidence_net_output'].quantile(0.25)
    
    high_conf_miou = df[df['confidence_net_output'] >= high_conf_threshold]['actual_miou'].mean()
    low_conf_miou = df[df['confidence_net_output'] <= low_conf_threshold]['actual_miou'].mean()
    
    print(f"\n🎯 Confidence vs Performance:")
    print(f"High confidence (top 25%): mIoU = {high_conf_miou:.3f}")
    print(f"Low confidence (bottom 25%): mIoU = {low_conf_miou:.3f}")
    print(f"Performance gap: {high_conf_miou - low_conf_miou:.3f}")
    
    return df

def analyze_class_wise_performance(df):
    """Analyze performance for each class"""
    print(f"\n🎯 Class-wise Analysis:")
    
    # Extract class-wise IoU columns
    class_columns = [col for col in df.columns if col.startswith('class_') and col.endswith('_iou')]
    
    class_performance = []
    for col in class_columns:
        class_id = int(col.split('_')[1])
        class_name = CLASS_INFO[class_id]['name']
        
        # Get non-null IoU values for this class
        class_ious = df[col].dropna()
        if len(class_ious) > 0:
            avg_iou = class_ious.mean()
            class_performance.append({
                'class_id': class_id,
                'class_name': class_name,
                'avg_iou': avg_iou,
                'samples_count': len(class_ious)
            })
    
    # Sort by average IoU
    class_performance.sort(key=lambda x: x['avg_iou'], reverse=True)
    
    print(f"\n🏆 Best performing classes:")
    for perf in class_performance[:5]:
        print(f"  {perf['class_name']}: IoU={perf['avg_iou']:.3f} ({perf['samples_count']} samples)")
    
    print(f"\n📉 Worst performing classes:")
    for perf in class_performance[-5:]:
        print(f"  {perf['class_name']}: IoU={perf['avg_iou']:.3f} ({perf['samples_count']} samples)")
    
    # Save class-wise analysis
    class_df = pd.DataFrame(class_performance)
    class_df.to_csv("class_wise_performance.csv", index=False)
    print(f"✅ Class-wise analysis saved to class_wise_performance.csv")

if __name__ == "__main__":
    # Test ConfidenceNet on training data
    results_df = test_confidence_net_on_train()
    
    # Analyze class-wise performance
    analyze_class_wise_performance(results_df)
    
    print(f"\n🎉 Analysis completed!")
    print(f"Check 'confidence_vs_miou_results.csv' for detailed results")
    print(f"Check 'class_wise_performance.csv' for class-wise analysis")

/tmp/ipykernel_1046869/4016134043.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  confidence_net.load_state_dict(torch.load('confidence_net_final.pth', map_location=dev

RuntimeError: Error(s) in loading state_dict for ConfidenceNet:
	Missing key(s) in state_dict: "prob_conv.0.weight", "prob_conv.0.bias", "prob_conv.2.weight", "prob_conv.2.bias", "point_conv.0.weight", "point_conv.0.bias", "point_conv.2.weight", "point_conv.2.bias", "combined_conv.0.weight", "combined_conv.0.bias", "combined_conv.2.weight", "combined_conv.2.bias", "fc.0.weight", "fc.0.bias", "fc.2.weight", "fc.2.bias", "fc.4.weight", "fc.4.bias". 
	Unexpected key(s) in state_dict: "probability_encoder.0.weight", "probability_encoder.0.bias", "probability_encoder.1.weight", "probability_encoder.1.bias", "probability_encoder.1.running_mean", "probability_encoder.1.running_var", "probability_encoder.1.num_batches_tracked", "probability_encoder.3.weight", "probability_encoder.3.bias", "probability_encoder.4.weight", "probability_encoder.4.bias", "probability_encoder.4.running_mean", "probability_encoder.4.running_var", "probability_encoder.4.num_batches_tracked", "point_encoder.0.weight", 

In [1]:
#to check confidence
import torch
import numpy as np
import cv2
import os
from glob import glob
import pandas as pd
from sklearn.metrics import confusion_matrix
import torch.nn.functional as F

def calculate_iou(pred, target, num_classes):
    """Calculate IoU for each class"""
    ious = []
    for class_id in range(num_classes):
        pred_mask = (pred == class_id)
        target_mask = (target == class_id)
        
        intersection = np.logical_and(pred_mask, target_mask).sum()
        union = np.logical_or(pred_mask, target_mask).sum()
        
        if union == 0:
            ious.append(1.0)  # If both are empty, IoU is 1
        else:
            ious.append(intersection / union)
    
    return ious

def calculate_miou(pred, target, num_classes):
    """Calculate mean IoU"""
    ious = calculate_iou(pred, target, num_classes)
    return np.mean(ious)

def test_confidence_net_on_train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load trained ConfidenceNet
    confidence_net = ConfidenceNet(num_classes=NUM_CLASSES)
    confidence_net.load_state_dict(torch.load('confidence_net_final.pth', map_location=device))
    confidence_net.to(device)
    confidence_net.eval()
    
    # Paths for your training data
    train_image_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
    train_mask_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_masks"
    train_point_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
    prediction_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_sam_predictions"
    
    # Get all training images
    image_files = sorted(glob(os.path.join(train_image_dir, "*.png")))
    print(f"Testing on {len(image_files)} training images...")
    
    results = []
    
    for i, image_path in enumerate(image_files):
        if i % 50 == 0:
            print(f"Processed {i}/{len(image_files)} images...")
        
        base_name = os.path.basename(image_path).split('.')[0]
        
        try:
            # Load pre-computed SAM predictions
            pred_path = os.path.join(prediction_dir, f"{base_name}_pred.npy")
            prob_path = os.path.join(prediction_dir, f"{base_name}_prob.npy")
            
            if not os.path.exists(pred_path) or not os.path.exists(prob_path):
                print(f"⚠️ Missing prediction files for {base_name}")
                continue
            
            prediction = np.load(pred_path)
            probability_maps = np.load(prob_path)
            
            # Load ground truth
            mask_path = os.path.join(train_mask_dir, f"{base_name}.png")
            ground_truth = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            if ground_truth is None:
                print(f"⚠️ Could not load ground truth for {base_name}")
                continue
            
            # Ensure same shape
            if ground_truth.shape != prediction.shape:
                ground_truth = cv2.resize(ground_truth, (prediction.shape[1], prediction.shape[0]), 
                                        interpolation=cv2.INTER_NEAREST)
            
            # Calculate actual mIoU
            miou = calculate_miou(prediction, ground_truth, NUM_CLASSES)
            
            # Calculate geometric features
            dataset = ConfidenceDataset(train_image_dir, train_mask_dir, train_point_dir, prediction_dir)
            geometric_features = dataset.calculate_geometric_features(prediction)
            point_maps = dataset.create_point_map(base_name, prediction.shape)
            
            # Convert to tensors
            probability_tensor = torch.from_numpy(probability_maps).float().unsqueeze(0).to(device)
            point_tensor = torch.from_numpy(point_maps).float().unsqueeze(0).to(device)
            geometric_tensor = torch.from_numpy(geometric_features).float().unsqueeze(0).to(device)
            
            # Get confidence scores from ConfidenceNet
            with torch.no_grad():
                confidence_scores = confidence_net(probability_tensor, point_tensor, geometric_tensor)
                confidence_map = confidence_scores.squeeze().cpu().numpy()
            
            # Calculate average confidence
            avg_confidence = np.mean(confidence_map)
            
            # Determine if confident (using threshold = 0.7)
            is_confident = "Yes" if avg_confidence > 0.7 else "No"
            
            # Store simplified results
            result = {
                'image_name': base_name,
                'miou': miou,
                'confidence_score': avg_confidence,
                'is_confident': is_confident
            }
            
            results.append(result)
            
            print(f"📊 {base_name}: mIoU={miou:.3f}, Confidence={avg_confidence:.3f}, Confident={is_confident}")
            
        except Exception as e:
            print(f"⚠️ Error processing {base_name}: {e}")
            continue
    
    # Create simplified DataFrame
    df = pd.DataFrame(results)
    
    # Calculate correlation between confidence and mIoU
    correlation = df['confidence_score'].corr(df['miou'])
    print(f"\n📈 Correlation between Confidence Score and mIoU: {correlation:.3f}")
    
    # Save simplified results
    output_csv = "confidence_vs_miou_simple.csv"
    df.to_csv(output_csv, index=False)
    print(f"✅ Simplified results saved to {output_csv}")
    
    # Print summary statistics
    print(f"\n📊 Summary Statistics:")
    print(f"Average Confidence Score: {df['confidence_score'].mean():.3f}")
    print(f"Average mIoU: {df['miou'].mean():.3f}")
    print(f"Confident images: {len(df[df['is_confident'] == 'Yes'])}/{len(df)}")
    print(f"Not confident images: {len(df[df['is_confident'] == 'No'])}/{len(df)}")
    
    # Analyze confident vs not-confident performance
    confident_miou = df[df['is_confident'] == 'Yes']['miou'].mean()
    not_confident_miou = df[df['is_confident'] == 'No']['miou'].mean()
    
    print(f"\n🎯 Performance by Confidence Level:")
    print(f"Confident images average mIoU: {confident_miou:.3f}")
    print(f"Not confident images average mIoU: {not_confident_miou:.3f}")
    print(f"Performance gap: {confident_miou - not_confident_miou:.3f}")
    
    return df

if __name__ == "__main__":
    # Test ConfidenceNet on training data
    results_df = test_confidence_net_on_train()
    
    print(f"\n🎉 Analysis completed!")
    print(f"Check 'confidence_vs_miou_simple_train.csv' for simplified results")

NameError: name 'ConfidenceNet' is not defined